# 🎵 Pipeline Profissional de Restauração de Áudio

Este notebook implementa um pipeline completo de restauração e masterização de áudio que inclui:

- ✅ **Análise Espectral Detalhada**
- ✅ **Redução de Ruído Avançada**
- ✅ **Restauração de Frequências**
- ✅ **Separação de Stems (opcional)**
- ✅ **Processamento Profissional**
- ✅ **Masterização Automatizada**

---

## 📦 Instalação de Dependências

In [ ]:
%%capture
# Instalar dependências necessárias
!pip install librosa soundfile scipy matplotlib numpy
!pip install noisereduce

# Opcional: Demucs para separação de stems de alta qualidade
# Descomente a linha abaixo se quiser usar Demucs (requer GPU e mais tempo)
# !pip install demucs

## 💾 Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Definir caminho para a pasta 00-restore no seu Drive
AUDIO_INPUT_DIR = '/content/drive/MyDrive/00-restore'
AUDIO_OUTPUT_DIR = '/content/drive/MyDrive/00-restore/restored_output'

print(f"✓ Drive montado")
print(f"Pasta de entrada: {AUDIO_INPUT_DIR}")
print(f"Pasta de saída: {AUDIO_OUTPUT_DIR}")

## 📥 Baixar Módulos do Pipeline

In [ ]:
# Clone o repositório (ajuste a URL conforme necessário)
!git clone https://github.com/guitorte/musicas.git /content/audio-pipeline-repo

# Copiar módulos para o ambiente de trabalho
import sys
import shutil
from pathlib import Path

# Adicionar ao path
sys.path.insert(0, '/content/audio-pipeline-repo/audio-restoration-pipeline')

print("✓ Módulos carregados")

## 🔧 Importar Módulos

In [ ]:
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Importar módulos do pipeline
from modules.pipeline import AudioRestorationPipeline
from modules.spectral_analysis import SpectralAnalyzer
from modules.frequency_restoration import FrequencyRestorer
from modules.stem_separation import StemSeparator
from modules.audio_processing import AudioProcessor

print("✓ Todos os módulos importados com sucesso!")

## ⚙️ Configuração do Pipeline

In [ ]:
# Configuração do pipeline
CONFIG = {
    # Limpeza básica
    'remove_clicks': True,              # Remover clicks e pops
    'reduce_noise': True,               # Reduzir ruído
    'noise_reduction_strength': 0.7,    # Força da redução (0-1)
    
    # Restauração de frequências
    'restore_frequencies': True,        # Restaurar frequências perdidas
    'freq_restoration_method': 'harmonic_synthesis',  # ou 'spectral_extension'
    'enhance_bass': False,              # Realçar graves
    'bass_enhancement_amount': 1.3,     # Quantidade de realce de graves
    'psychoacoustic_enhancement': True, # Melhorias psicoacústicas
    
    # Separação de stems (AVISO: consome muita memória e tempo)
    'separate_stems': False,            # Separar em stems (vocal, drums, bass, other)
    'stem_separation_model': 'basic',   # 'basic' ou 'demucs' (requer GPU)
    'process_stems_individually': False,# Processar cada stem separadamente
    
    # Masterização
    'target_lufs': -14.0,               # LUFS alvo (-14 para streaming, -16 para broadcast)
    'master_eq': {                      # EQ de masterização (em dB)
        'bass': 0.5,                    # 60-250 Hz
        'mid': 0.0,                     # 500-2000 Hz
        'presence': 1.0,                # 4000-6000 Hz
        'treble': 0.8                   # 6000-20000 Hz
    },
    'add_presence': True                # Adicionar brilho e presença
}

print("✓ Configuração definida")
print("\nConfiguração atual:")
import json
print(json.dumps(CONFIG, indent=2))

## 📂 Listar Arquivos de Áudio

In [ ]:
# Buscar todos os arquivos MP3 e WAV
audio_files = []
audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, '*.mp3')))
audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, '*.wav')))
audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, '*.MP3')))
audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, '*.WAV')))

audio_files = sorted(set(audio_files))  # Remover duplicatas e ordenar

print(f"✓ Encontrados {len(audio_files)} arquivos de áudio:\n")
for i, file in enumerate(audio_files, 1):
    file_size_mb = os.path.getsize(file) / (1024 * 1024)
    print(f"{i:2d}. {Path(file).name} ({file_size_mb:.2f} MB)")

if not audio_files:
    print(f"⚠️ AVISO: Nenhum arquivo encontrado em {AUDIO_INPUT_DIR}")
    print("Verifique se o caminho está correto e se há arquivos MP3 ou WAV na pasta.")

## 🚀 Inicializar Pipeline

In [ ]:
# Criar diretório de saída
os.makedirs(AUDIO_OUTPUT_DIR, exist_ok=True)

# Inicializar pipeline
pipeline = AudioRestorationPipeline(
    sr=44100,
    output_base_dir=AUDIO_OUTPUT_DIR,
    log_dir=os.path.join(AUDIO_OUTPUT_DIR, 'logs')
)

print("✓ Pipeline inicializado")

## 🎯 Processar Arquivo Individual (Teste)

In [ ]:
# Processar apenas o primeiro arquivo como teste
if audio_files:
    test_file = audio_files[0]
    print(f"Processando arquivo de teste: {Path(test_file).name}")
    print("="*70)
    
    result = pipeline.process_audio(
        test_file,
        config=CONFIG
    )
    
    print("\n" + "="*70)
    print("✓ TESTE COMPLETO!")
    print(f"\nArquivo final: {result['stages']['mastering']['output']}")
    print(f"Pasta de saída: {result['output_dir']}")
else:
    print("⚠️ Nenhum arquivo para processar")

## 📊 Visualizar Análise do Arquivo Processado

In [ ]:
# Mostrar visualização da análise
if audio_files:
    from IPython.display import Image, display
    
    viz_path = os.path.join(result['output_dir'], 'analysis_visualization.png')
    if os.path.exists(viz_path):
        print("Análise Espectral:")
        display(Image(filename=viz_path))
    
    # Mostrar análise JSON
    analysis_path = os.path.join(result['output_dir'], 'analysis.json')
    if os.path.exists(analysis_path):
        import json
        with open(analysis_path, 'r') as f:
            analysis = json.load(f)
        
        print("\nRecomendações da Análise:")
        if analysis.get('recommendations'):
            for rec in analysis['recommendations']:
                print(f"  [{rec['severity'].upper()}] {rec['message']}")
        else:
            print("  ✓ Nenhum problema crítico detectado")

## 🔄 Processar Todos os Arquivos em Batch

**⚠️ AVISO:** Isso pode levar bastante tempo dependendo da quantidade e tamanho dos arquivos!

In [ ]:
# Descomente a linha abaixo para processar TODOS os arquivos
# BATCH_PROCESS = True
BATCH_PROCESS = False  # Manter False por segurança

if BATCH_PROCESS and audio_files:
    print(f"Iniciando processamento em batch de {len(audio_files)} arquivos...")
    print("Isso pode levar vários minutos ou horas!\n")
    
    batch_results = pipeline.batch_process(
        audio_files,
        config=CONFIG
    )
    
    # Resumo
    successful = sum(1 for r in batch_results if 'error' not in r)
    failed = len(batch_results) - successful
    
    print(f"\n{'='*70}")
    print(f"BATCH COMPLETO!")
    print(f"Sucesso: {successful}/{len(batch_results)}")
    if failed > 0:
        print(f"Falhas: {failed}")
    print(f"{'='*70}")
else:
    print("Processamento em batch desabilitado.")
    print("Para processar todos os arquivos, mude BATCH_PROCESS = True na célula acima.")

## 🎛️ Processamento Customizado (Avançado)

Use esta seção para processar arquivos com configurações personalizadas

In [ ]:
# Exemplo: Processar arquivo específico com configurações customizadas

# Selecionar arquivo (ajuste o índice conforme necessário)
# custom_file = audio_files[0]  # Primeiro arquivo

# Configuração customizada
custom_config = {
    'remove_clicks': True,
    'reduce_noise': True,
    'noise_reduction_strength': 0.8,    # Redução mais agressiva
    'restore_frequencies': True,
    'freq_restoration_method': 'spectral_extension',  # Método alternativo
    'enhance_bass': True,               # Ativar realce de graves
    'bass_enhancement_amount': 1.5,     # Realce mais forte
    'psychoacoustic_enhancement': True,
    'separate_stems': False,
    'target_lufs': -16.0,               # Mais silencioso (para broadcast)
    'master_eq': {
        'bass': 1.0,                    # Mais graves
        'mid': -0.5,                    # Reduzir médios
        'presence': 1.5,                # Mais presença
        'treble': 1.2                   # Mais brilho
    },
    'add_presence': True
}

# Descomente para processar
# result = pipeline.process_audio(custom_file, config=custom_config)
# print(f"Processado: {result['stages']['mastering']['output']}")

print("✓ Seção de processamento customizado pronta")
print("Descomente as linhas acima para processar com configurações personalizadas")

## 💾 Download de Resultados

Os arquivos processados estão salvos em:
- **Google Drive:** `00-restore/restored_output/`

Você pode acessá-los diretamente pelo Drive ou fazer download abaixo:

In [ ]:
from google.colab import files

# Listar arquivos finais processados
output_files = glob.glob(os.path.join(AUDIO_OUTPUT_DIR, '**/*FINAL.wav'), recursive=True)

if output_files:
    print(f"Encontrados {len(output_files)} arquivos finais:\n")
    for i, file in enumerate(output_files, 1):
        print(f"{i}. {Path(file).name}")
    
    # Descomente para fazer download do primeiro arquivo
    # files.download(output_files[0])
    
    print("\n💡 Descomente a linha 'files.download()' acima para baixar um arquivo")
else:
    print("Nenhum arquivo final encontrado ainda.")

## 🔧 Utilitários Extras

In [ ]:
# Função para comparar antes/depois
def compare_audio(original_path, processed_path):
    """Compara áudio original vs processado"""
    import IPython.display as ipd
    from pathlib import Path
    
    print("🔊 ORIGINAL:")
    print(f"   {Path(original_path).name}")
    display(ipd.Audio(original_path))
    
    print("\n🎵 PROCESSADO:")
    print(f"   {Path(processed_path).name}")
    display(ipd.Audio(processed_path))

# Exemplo de uso:
# compare_audio(audio_files[0], result['stages']['mastering']['output'])

print("✓ Utilitários carregados")
print("Use compare_audio(original, processado) para comparar arquivos")

---

## 📝 Notas

- O pipeline salva múltiplas versões do áudio durante o processamento
- Análises detalhadas são salvas em JSON e PNG
- O arquivo final masterizado tem sufixo `_FINAL.wav`
- Logs completos estão disponíveis na pasta `logs/`

## 🎯 Dicas de Uso

1. **Sempre teste com um arquivo primeiro** antes de processar em batch
2. **Ajuste as configurações** baseado nos resultados da análise
3. **Redução de ruído agressiva** (>0.8) pode criar artefatos
4. **Separação de stems** é muito mais precisa com Demucs, mas requer GPU
5. **LUFS -14** é padrão para Spotify/YouTube, **-16** para TV/Rádio

---

**Pipeline desenvolvido com tecnologias de ponta em processamento de áudio**

🎵 Bom trabalho!